## Data Transformation

First we transform some variables to make it easier for our models to extract the right information. The month variable is converted into sinus and cosine values to allow for a cyclical interpretation. A one-hot encoding or simply categorical approach would miss for example the fact that December and January are very close to each other. Further, we one-hot encode the categorical feature *Vegetation*. The first dummy is not dropped here as most models we use allow this. For regression based models one of these dummies should be removed before use. Finally, we normalize our data. While this may not be necessary for every model we use it does not hurt either, so we simply do it for all. Here, we keep the categorical vegetation variable for now as some variables do not need one-hot-encoding.

In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('../data/processed/FW_Veg_Rem_Combined_cleaned.csv')
df.drop(columns=['fire_size','fire_size_class'], inplace=True)

month_map = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
if df["discovery_month"].dtype == "object":
    df["discovery_month"] = df["discovery_month"].map(month_map)

# Create sine/cosine features
df["month_sin"] = np.sin(2 * np.pi * df["discovery_month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["discovery_month"] / 12)

# Drop original month column
df.drop("discovery_month", axis=1, inplace=True)

# Convert categorical variables to one-hot encoding
# dropping the first category not necessary for neural networks
df_2 = pd.get_dummies(df, columns=["Vegetation"], drop_first=False)

vegetation_map = {
    0: "Unknown",  # Assuming 0 means something specific or is a placeholder
    4: "Temperate Evergreen Needleleaf Forest",
    9: "C3 Grassland/Steppe",
    12: "Open Shrubland",
    14: "Desert",
    15: "Polar Desert/Rock/Ice",
    16: "Secondary Tropical Evergreen Broadleaf Forest"
}

# Add original vegetation column with categorical names
df_2["Vegetation"] = df["Vegetation"].map(vegetation_map)
df_2.head()

,index,latitude,longitude,Temp_pre_30,Temp_pre_15,Temp_pre_7,Wind_pre_30,Wind_pre_15,Wind_pre_7,Hum_pre_30,...,month_sin,month_cos,Vegetation_0,Vegetation_4,Vegetation_9,Vegetation_12,Vegetation_14,Vegetation_15,Vegetation_16,Vegetation
0,0,18.105072,-66.753044,24.480974,24.716923,24.902597,4.341807,3.492857,3.262092,78.216590,...,8.660254e-01,0.500000,False,False,False,True,False,False,False,Open Shrubland
1,1,35.038330,-87.610000,7.553433,7.010000,0.343529,2.709764,2.881707,1.976471,70.840000,...,-2.449294e-16,1.000000,False,False,False,False,False,True,False,Polar Desert/Rock/Ice
2,2,34.947800,-88.722500,4.971930,5.782766,5.558750,3.364499,2.923830,2.695833,75.531629,...,8.660254e-01,0.500000,False,False,False,False,False,False,True,Secondary Tropical Evergreen Broadleaf Forest
3,3,39.641400,-119.308300,16.275967,18.996181,18.142564,4.054982,3.398329,3.671282,44.778429,...,1.224647e-16,-1.000000,True,False,False,False,False,False,False,Unknown
4,4,31.316978,-83.393649,14.877341,16.409326,17.175319,2.000214,1.727202,1.590696,79.896679,...,5.000000e-01,0.866025,False,False,False,True,False,False,False,Open Shrubland


## Data Load

The standardization is to be applied separately for each model as the standardizer should never be fitted on the validation data for each of the k Folds used in cross validation.

In [7]:
# load df_2 as a csv file into the data/processed folder
df_2.to_csv('../data/processed/FW_Veg_Rem_Combined_transformed.csv', index=False)